# Phase 5 — SHAP Meta-Explanation

Explains which dataset properties determine when each clustering method wins.
Uses KernelExplainer on a k=5 kNN (smoother probabilities than deployed k=1).
The k=5 model is for explanation only — not used for accuracy reporting.

**Outputs**: shap_global_importance.png, shap_beeswarm_global.png,
shap_per_method.png, shap_heatmap.png, shap_dependence_*.png, shap_values.csv

In [ ]:
import os, sys, pickle, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.neighbors import KNeighborsClassifier

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if os.path.join(ROOT, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(ROOT, 'src'))
from meta_learner import LSE_COLS, META_COLS

META_DIR   = os.path.join(ROOT, 'data', 'meta_table')
MODELS_DIR = os.path.join(ROOT, 'outputs', 'models')
FIGS_DIR   = os.path.join(ROOT, 'outputs', 'figures')
os.makedirs(FIGS_DIR, exist_ok=True)

SEED = 42
np.random.seed(SEED)

METHOD_COLORS = {
    'kmeans'   : '#4C72B0',
    'dbscan'   : '#DD8452',
    'agg'      : '#55A868',
    'gmm'      : '#C44E52',
    'autoenc'  : '#8172B2',
    'dictlearn': '#937860',
}
print(f'shap {shap.__version__}')

In [ ]:
df = pd.read_csv(os.path.join(META_DIR, 'meta_training_optA.csv'))
df = df.dropna(subset=['best_method']).reset_index(drop=True)

feat_cols = [c for c in df.columns if c not in META_COLS + LSE_COLS]
X_raw = df[feat_cols].values.astype(float)
y     = df['best_method'].values

with open(os.path.join(MODELS_DIR, 'meta_clf_optA.pkl'), 'rb') as f:
    saved = pickle.load(f)
pipe_k1 = saved['pipeline']

imputer = pipe_k1.named_steps['impute']
scaler  = pipe_k1.named_steps['scale']
X_proc  = scaler.transform(imputer.transform(X_raw))

print(f'X_proc shape : {X_proc.shape}')
print(f'Classes      : {pipe_k1.classes_}')
print(pd.Series(y).value_counts().to_string())

In [ ]:
# k=5 kNN for smoother predict_proba (explanation only, not accuracy reporting)
knn5 = KNeighborsClassifier(n_neighbors=5)
knn5.fit(X_proc, y)
print('k=5 fitted. Classes:', knn5.classes_)
print(pd.DataFrame(knn5.predict_proba(X_proc[:3]), columns=knn5.classes_).round(3))

## Compute SHAP Values

In [ ]:
background = shap.kmeans(X_proc, 15)
explainer  = shap.KernelExplainer(knn5.predict_proba, background)

print('Computing SHAP values (51 samples x 6 classes) ...')
shap_values = explainer.shap_values(X_proc, nsamples=300, silent=False)
# list of 6 arrays each (51, 18)
classes  = list(knn5.classes_)
shap_arr = np.array(shap_values)

# SHAP 0.41+ may return (n_samples, n_features, n_classes) instead of
# a list giving (n_classes, n_samples, n_features). Normalise to (6, 51, 18).
n, p, k = len(X_proc), len(feat_cols), len(classes)
if shap_arr.shape == (n, p, k):
    shap_arr = shap_arr.transpose(2, 0, 1)
elif shap_arr.shape != (k, n, p):
    raise ValueError(f'Unexpected shap_arr shape: {shap_arr.shape}')

print(f'Done. shap_arr shape (classes, samples, features): {shap_arr.shape}')

In [ ]:
rows = []
for ci, cls in enumerate(classes):
    for si in range(len(X_proc)):
        row = {'class': cls,
               'dataset_id': int(df['dataset_id'].iloc[si]),
               'true_method': y[si]}
        for fi, fn in enumerate(feat_cols):
            row[fn] = float(shap_arr[ci, si, fi])
        rows.append(row)
shap_df = pd.DataFrame(rows)
shap_path = os.path.join(META_DIR, 'shap_values.csv')
shap_df.to_csv(shap_path, index=False)
print(f'Saved -> {shap_path}  shape={shap_df.shape}')

## Global Feature Importance

In [ ]:
global_importance = np.abs(shap_arr).mean(axis=(0, 1))  # (18,)
imp_df = pd.DataFrame({'feature': feat_cols, 'mean_abs_shap': global_importance})
imp_df = imp_df.sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

print('=== Global feature importance ===')
print(imp_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(imp_df['feature'][::-1], imp_df['mean_abs_shap'][::-1], color='steelblue')
ax.set_xlabel('Mean |SHAP value|')
ax.set_title('Global meta-feature importance\n(higher = more influence on method selection)')
plt.tight_layout()
path = os.path.join(FIGS_DIR, 'shap_global_importance.png')
fig.savefig(path, dpi=130)
plt.close()
print(f'Saved -> {path}')

## Beeswarm — Top Features Overall

In [ ]:
shap_mean_classes = shap_arr.mean(axis=0)  # (51, 18)

shap_expl = shap.Explanation(
    values=shap_mean_classes,
    base_values=np.zeros(len(X_proc)),
    data=X_proc,
    feature_names=feat_cols,
)
fig, ax = plt.subplots(figsize=(9, 7))
shap.plots.beeswarm(shap_expl, max_display=18, show=False)
plt.title('SHAP beeswarm — mean across all methods\n(each dot = one dataset)')
plt.tight_layout()
path = os.path.join(FIGS_DIR, 'shap_beeswarm_global.png')
fig.savefig(path, dpi=130, bbox_inches='tight')
plt.close()
print(f'Saved -> {path}')

## Per-Method SHAP Importance

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.ravel()
per_class_top = {}

for ci, cls in enumerate(classes):
    sv       = shap_arr[ci]             # (51, 18)
    mean_abs = np.abs(sv).mean(0)       # (18,)
    order    = np.argsort(mean_abs)
    top_feats = [feat_cols[i] for i in order[-8:]]
    top_vals  = mean_abs[order[-8:]]
    per_class_top[cls] = list(zip(top_feats[::-1], top_vals[::-1]))

    ax = axes[ci]
    ax.barh(top_feats, top_vals, color=METHOD_COLORS.get(cls, 'steelblue'), alpha=0.85)
    ax.set_title(f'-> {cls}  (n_best={int((y==cls).sum())})', fontweight='bold')
    ax.set_xlabel('Mean |SHAP|')
    ax.tick_params(axis='y', labelsize=8)

plt.suptitle('Per-method feature importance', fontsize=11, y=1.01)
plt.tight_layout()
path = os.path.join(FIGS_DIR, 'shap_per_method.png')
fig.savefig(path, dpi=130, bbox_inches='tight')
plt.close()
print(f'Saved -> {path}')

print('\n=== Top-3 features per method ===')
for cls, pairs in per_class_top.items():
    top3 = ', '.join(f'{f} ({v:.4f})' for f, v in pairs[:3])
    print(f'  {cls:12s}: {top3}')

## Feature x Method Heatmap

In [ ]:
heat = np.zeros((len(feat_cols), len(classes)))
for ci, cls in enumerate(classes):
    heat[:, ci] = shap_arr[ci].mean(axis=0)

heat_df = pd.DataFrame(heat, index=feat_cols, columns=classes)
heat_df = heat_df.loc[imp_df['feature'].tolist()]

fig, ax = plt.subplots(figsize=(10, 8))
vmax = np.abs(heat).max()
im   = ax.imshow(heat_df.values, aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
plt.colorbar(im, ax=ax, label='Mean SHAP (positive = pushes toward method)')
ax.set_xticks(range(len(classes)))
ax.set_xticklabels(classes, rotation=30, ha='right', fontsize=10)
ax.set_yticks(range(len(feat_cols)))
ax.set_yticklabels(imp_df['feature'].tolist(), fontsize=9)
ax.set_title('Mean SHAP: which features steer toward each method\n(red=toward, blue=away)', fontsize=11)
plt.tight_layout()
path = os.path.join(FIGS_DIR, 'shap_heatmap.png')
fig.savefig(path, dpi=130, bbox_inches='tight')
plt.close()
print(f'Saved -> {path}')
print(heat_df.round(4).to_string())

## Dependence Plots — Top 4 Features

In [ ]:
top4 = imp_df['feature'].head(4).tolist()

for feat in top4:
    fi = feat_cols.index(feat)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    feat_vals        = X_proc[:, fi]
    shap_vals_global = shap_mean_classes[:, fi]
    ax1.scatter(feat_vals, shap_vals_global,
                c=[METHOD_COLORS.get(m, 'gray') for m in y],
                alpha=0.8, s=60, edgecolors='w', linewidths=0.4)
    ax1.axhline(0, ls='--', color='gray', lw=0.8)
    ax1.set_xlabel(f'{feat} (scaled)')
    ax1.set_ylabel('Mean SHAP value')
    ax1.set_title(f'{feat} - global effect')
    handles = [Line2D([0],[0], marker='o', color='w',
                      markerfacecolor=v, markersize=7, label=k)
               for k, v in METHOD_COLORS.items()]
    ax1.legend(handles=handles, fontsize=7)

    bins = np.percentile(feat_vals, [0,20,40,60,80,100])
    bin_centers = 0.5*(bins[:-1]+bins[1:])
    for ci, cls in enumerate(classes):
        sv_cls = shap_arr[ci][:, fi]
        bm = [sv_cls[(feat_vals>=bins[b])&(feat_vals<bins[b+1])].mean()
              if ((feat_vals>=bins[b])&(feat_vals<bins[b+1])).any() else np.nan
              for b in range(len(bins)-1)]
        ax2.plot(bin_centers, bm, marker='o', label=cls,
                 color=METHOD_COLORS.get(cls, 'gray'), lw=1.5, ms=5)
    ax2.axhline(0, ls='--', color='gray', lw=0.8)
    ax2.set_xlabel(f'{feat} (scaled, quintiles)')
    ax2.set_ylabel('Mean SHAP per bin')
    ax2.set_title(f'{feat} - per-method effect')
    ax2.legend(fontsize=7)

    plt.suptitle(f'Dependence: {feat}', fontsize=12)
    plt.tight_layout()
    path = os.path.join(FIGS_DIR, f'shap_dependence_{feat}.png')
    fig.savefig(path, dpi=130, bbox_inches='tight')
    plt.close()
    print(f'Saved -> {path}')

## Key Findings

In [ ]:
df['lse_std'] = df[LSE_COLS].std(axis=1)
lse_corr = df[feat_cols+['lse_std']].corr()['lse_std'].drop('lse_std').abs()

summary = imp_df.copy()
summary['lse_std_corr'] = summary['feature'].map(lse_corr)
summary['rank_shap']    = range(1, len(summary)+1)
summary['rank_corr']    = summary['lse_std_corr'].rank(ascending=False).astype(int)

print('=== SHAP rank vs LSE-variance correlation rank ===')
print(summary[['feature','mean_abs_shap','lse_std_corr','rank_shap','rank_corr']]
      .round(4).to_string(index=False))

agreed = summary[(summary['rank_shap']<=6)&(summary['rank_corr']<=6)]
print(f'\nFeatures in top-6 by BOTH metrics (most reliable drivers):')
print('  '+', '.join(agreed['feature'].tolist()))

print('\n=== Summary ===')
print(f"  Top feature    : {imp_df['feature'].iloc[0]}")
print(f"  Top 3          : {', '.join(imp_df['feature'].head(3).tolist())}")
print(f"  Bottom 3       : {', '.join(imp_df['feature'].tail(3).tolist())}")
print()
print('  Per-method key driver:')
for cls, pairs in per_class_top.items():
    print(f'    {cls:12s} -> {pairs[0][0]} ({pairs[0][1]:.4f})')
print()
print('Phase 5 complete.')